# EarthMover Regularizer: Numba/Numpy Init Benchmark

This notebook adds a new `EarthMoversDistanceRegularizerNumbaInit` class that moves one-time initialization work from TensorFlow ops to Numba/Numpy.

It runs:
- Init-time benchmark vs current `EarthMoversDistanceRegularizer`
- Numerical equivalence checks on multiple inputs
- Optional call-time benchmark (runtime path)


In [1]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer

from v1_model_utils import load_sparse, other_v1_utils
from v1_model_utils.loss_functions import EarthMoversDistanceRegularizer

try:
    from numba import njit
    HAS_NUMBA = True
except Exception:
    HAS_NUMBA = False

print(f"TensorFlow: {tf.__version__}")
print(f"Numba available: {HAS_NUMBA}")


2026-02-12 13:32:25.636893: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 13:32:25.674121: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-12 13:32:25.674169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-12 13:32:25.675357: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-12 13:32:25.682015: I tensorflow/core/platform/cpu_feature_guar

TensorFlow: 2.15.0
Numba available: True


In [2]:
if HAS_NUMBA:
    @njit(cache=True)
    def _build_group_order_numba(group_ids, n_groups):
        """Stable bucket ordering by group id with CSR row_splits."""
        n = group_ids.shape[0]
        counts = np.zeros(n_groups, dtype=np.int64)
        for i in range(n):
            counts[group_ids[i]] += 1

        row_splits = np.empty(n_groups + 1, dtype=np.int64)
        row_splits[0] = 0
        for g in range(n_groups):
            row_splits[g + 1] = row_splits[g] + counts[g]

        write_ptr = np.empty(n_groups, dtype=np.int64)
        for g in range(n_groups):
            write_ptr[g] = row_splits[g]

        order = np.empty(n, dtype=np.int64)
        for i in range(n):
            g = group_ids[i]
            pos = write_ptr[g]
            order[pos] = i
            write_ptr[g] = pos + 1

        return order, row_splits

    @njit(cache=True)
    def _sort_initial_values_by_group_numba(initial_values, order, row_splits):
        """Sort initial values independently inside each group segment."""
        out = np.empty(initial_values.shape[0], dtype=initial_values.dtype)
        n_groups = row_splits.shape[0] - 1
        for g in range(n_groups):
            start = row_splits[g]
            end = row_splits[g + 1]
            size = end - start
            buf = np.empty(size, dtype=initial_values.dtype)
            for j in range(size):
                buf[j] = initial_values[order[start + j]]
            buf.sort()
            for j in range(size):
                out[start + j] = buf[j]
        return out


def _build_group_order_numpy(group_ids, n_groups):
    order = np.argsort(group_ids, kind='stable')
    counts = np.bincount(group_ids, minlength=n_groups)
    row_splits = np.empty(n_groups + 1, dtype=np.int64)
    row_splits[0] = 0
    np.cumsum(counts, dtype=np.int64, out=row_splits[1:])
    return order.astype(np.int64, copy=False), row_splits


def _sort_initial_values_by_group_numpy(initial_values, order, row_splits):
    out = np.empty(initial_values.shape[0], dtype=initial_values.dtype)
    for g in range(row_splits.shape[0] - 1):
        start = row_splits[g]
        end = row_splits[g + 1]
        out[start:end] = np.sort(initial_values[order[start:end]])
    return out


class EarthMoversDistanceRegularizerNumbaInit(Layer):
    """
    EMD regularizer with one-time setup moved to Numba/Numpy.
    Runtime call path is intentionally equivalent to the original implementation.
    """

    def __init__(self, strength, network, dtype=tf.float32):
        super().__init__()
        self._strength = tf.cast(strength, dtype)
        self._dtype = dtype

        voltage_scale = (network['node_params']['V_th'] - network['node_params']['E_L']).astype(np.float32)
        indices = np.asarray(network['synapses']['indices'])
        initial_value_np = np.asarray(network['synapses']['weights'], dtype=np.float32).copy()
        edge_type_ids_np = other_v1_utils.connection_type_ids(network).astype(np.int64, copy=False)
        initial_value_np /= voltage_scale[np.asarray(network['node_type_ids'])[indices[:, 0]]]

        _, group_ids_np = np.unique(edge_type_ids_np, return_inverse=True)
        n_groups = int(np.max(group_ids_np) + 1) if group_ids_np.size else 0

        if group_ids_np.size == 0:
            order_np = np.empty((0,), dtype=np.int64)
            row_splits_np = np.zeros((1,), dtype=np.int64)
            sorted_initial_flat_np = np.empty((0,), dtype=np.float32)
        else:
            if HAS_NUMBA:
                order_np, row_splits_np = _build_group_order_numba(group_ids_np, n_groups)
                sorted_initial_flat_np = _sort_initial_values_by_group_numba(initial_value_np, order_np, row_splits_np)
            else:
                order_np, row_splits_np = _build_group_order_numpy(group_ids_np, n_groups)
                sorted_initial_flat_np = _sort_initial_values_by_group_numpy(initial_value_np, order_np, row_splits_np)

        if order_np.size <= np.iinfo(np.int32).max:
            order_tf = tf.convert_to_tensor(order_np, dtype=tf.int32)
        else:
            order_tf = tf.convert_to_tensor(order_np, dtype=tf.int64)
        row_splits_tf = tf.convert_to_tensor(row_splits_np, dtype=tf.int32)
        sorted_initial_tf = tf.convert_to_tensor(sorted_initial_flat_np, dtype=self._dtype)

        self.num_unique = tf.constant(n_groups, dtype=tf.int32)
        self._group_indices = tf.RaggedTensor.from_row_splits(order_tf, row_splits_tf, validate=False)
        self._sorted_initial_values = tf.RaggedTensor.from_row_splits(sorted_initial_tf, row_splits_tf, validate=False)

    @tf.function(jit_compile=False)
    def __call__(self, x):
        if x.dtype != self._dtype:
            x = tf.cast(x, self._dtype)
        if len(x.shape) > 1 and x.shape[1] == 1:
            x = tf.squeeze(x, axis=1)

        emd_losses = tf.TensorArray(self._dtype, size=self.num_unique)
        for i in tf.range(self.num_unique):
            x_i = tf.gather(x, self._group_indices[i])
            y_i = self._sorted_initial_values[i]
            emd = tf.reduce_mean(tf.abs(tf.sort(x_i) - y_i))
            emd_losses = emd_losses.write(i, emd)
        reg_loss = tf.reduce_mean(emd_losses.stack())
        return reg_loss * self._strength


In [3]:
# Benchmark configuration
import os

data_dir_candidates = [
    'GLIF_network',
    'GLIF_network_nll',
    'GLIF_network_nll_full',
    'GLIF_network_nll_core',
]
DATA_DIR = next((d for d in data_dir_candidates if os.path.isdir(d)), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'None of the expected data dirs were found: {data_dir_candidates}')

N_NEURONS = 1200
SEED = 3000
STRENGTH = 0.1
DTYPE = tf.float32

print('Loading network...')
network = load_sparse.load_network(
    data_dir=DATA_DIR,
    core_only=False,
    n_neurons=N_NEURONS,
    seed=SEED,
    connected_selection=True,
    tensorflow_speed_up=False,
    random_weights=False,
    uniform_weights=False,
)
network['data_dir'] = DATA_DIR

print(f"n_nodes: {network['n_nodes']}")
print(f"n_edges: {network['n_edges']}")


Loading network...
Loading network_dat.pkl file...
> Maximum sample radius: 54.63
> Number of Neurons: 1200
> Number of Synapses: 55545
n_nodes: 1200
n_edges: 55545


In [14]:
def benchmark_init(cls, n_runs=5):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = cls(STRENGTH, network, dtype=DTYPE)
        times.append(time.perf_counter() - t0)
    return np.array(times, dtype=np.float64)


# Warmup: include one throwaway construction for each class.
old = EarthMoversDistanceRegularizer(STRENGTH, network, dtype=DTYPE)
new = EarthMoversDistanceRegularizerNumbaInit(STRENGTH, network, dtype=DTYPE)

n_runs = 5
orig_init = benchmark_init(EarthMoversDistanceRegularizer, n_runs=n_runs)
new_init = benchmark_init(EarthMoversDistanceRegularizerNumbaInit, n_runs=n_runs)

print('Init benchmark (seconds):')
print(f'  Original: mean={orig_init.mean():.4f}, std={orig_init.std():.4f}, runs={orig_init}')
print(f'  NumbaInit: mean={new_init.mean():.4f}, std={new_init.std():.4f}, runs={new_init}')
print(f'  Speedup: {orig_init.mean() / new_init.mean():.3f}x')


Init benchmark (seconds):
  Original: mean=0.8484, std=0.0494, runs=[0.8225313  0.83372965 0.89251863 0.91501697 0.77828989]
  NumbaInit: mean=0.0235, std=0.0009, runs=[0.02535778 0.0232421  0.02302062 0.02297256 0.02297671]
  Speedup: 36.081x


In [18]:
old._initial_value

<tf.Tensor: shape=(55545,), dtype=float32, numpy=
array([ 0.27618176, -0.9265097 , -0.37325215, ..., 10.30463   ,
       -0.08037385, -0.07738984], dtype=float32)>

In [15]:
new._sorted_initial_values

<tf.RaggedTensor [[-0.37325215, -0.37303245, -0.17619129, -0.166398, -0.16631223,
  -0.16623737, -0.16619809, -0.16258185, -0.16253233, -0.16207488,
  -0.13186823, -0.12251794, -0.12223984, -0.12211589, -0.12050997,
  -0.1201245, -0.120117836, -0.12000669, -0.09787851, -0.09783128,
  -0.081905074, -0.08188178, -0.08185737, -0.08185302, -0.08183038,
  -0.081728995, -0.07252671, -0.07247541, -0.07242788, -0.07235723,
  -0.072350636, -0.07232754, -0.06582654, -0.053626332, -0.05358949,
  -0.053571418, -0.05356635, -0.035719533, -0.035711177]            ,
 [0.27618176, 0.27643147, 0.27658308, 0.27679196, 0.27679232, 0.27684924,
  0.27699023, 0.27708158, 0.27710927, 0.27716926, 0.27741125, 0.4018576,
  0.40196347, 0.4020915, 0.40215608, 0.40225127, 0.402289, 0.40239027,
  0.40245044, 0.402479, 0.40273288, 0.40285355, 0.40299675, 0.50950056,
  0.5097073, 0.5100303, 0.51008064, 0.5103817, 0.5104372, 0.510587,
  0.54254687, 0.5425562, 0.5426307, 0.54300606, 0.5433784, 0.5434326,
  0.54356396, 

In [8]:
# Equivalence checks
orig = EarthMoversDistanceRegularizer(STRENGTH, network, dtype=DTYPE)
new = EarthMoversDistanceRegularizerNumbaInit(STRENGTH, network, dtype=DTYPE)

rng = np.random.default_rng(123)
n_edges = network['n_edges']
test_inputs = [
    tf.convert_to_tensor(rng.normal(size=n_edges).astype(np.float32)),
    tf.convert_to_tensor(rng.normal(loc=0.5, scale=2.0, size=n_edges).astype(np.float32)),
    tf.convert_to_tensor(rng.uniform(low=-1.0, high=1.0, size=n_edges).astype(np.float32)),
    tf.convert_to_tensor(np.asarray(network['synapses']['weights'], dtype=np.float32)),
]

diffs = []
for i, x in enumerate(test_inputs):
    y_orig = orig(x)
    y_new = new(x)
    abs_diff = float(tf.abs(y_orig - y_new).numpy())
    diffs.append(abs_diff)
    print(f'test[{i}] -> original={float(y_orig.numpy()):.8f}, new={float(y_new.numpy()):.8f}, abs_diff={abs_diff:.3e}')

max_abs_diff = float(np.max(diffs))
print(f'\nMax abs diff: {max_abs_diff:.3e}')
np.testing.assert_allclose(max_abs_diff, 0.0, rtol=0.0, atol=1e-6)
print('Equivalence check PASSED (atol=1e-6).')


test[0] -> original=0.07392868, new=0.07392868, abs_diff=0.000e+00
test[1] -> original=0.15216357, new=0.15216357, abs_diff=0.000e+00
test[2] -> original=0.04869141, new=0.04869141, abs_diff=0.000e+00
test[3] -> original=0.81481761, new=0.81481761, abs_diff=0.000e+00

Max abs diff: 0.000e+00
Equivalence check PASSED (atol=1e-6).


In [10]:
# Optional: runtime-call benchmark (after tracing)
x_bench = test_inputs[0]
_ = orig(x_bench)
_ = new(x_bench)

def benchmark_call(obj, x, n_runs=20):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = obj(x)
        times.append(time.perf_counter() - t0)
    return np.array(times, dtype=np.float64)

orig_call = benchmark_call(orig, x_bench, n_runs=20)
new_call = benchmark_call(new, x_bench, n_runs=20)

print('Call benchmark (seconds):')
print(f'  Original: mean={orig_call.mean():.6f}, std={orig_call.std():.6f}')
print(f'  NumbaInit: mean={new_call.mean():.6f}, std={new_call.std():.6f}')


Call benchmark (seconds):
  Original: mean=0.026717, std=0.001172
  NumbaInit: mean=0.025615, std=0.000842


## Notes

- If first-run init of `NumbaInit` is slower, that is expected due to Numba JIT compilation.
- Subsequent constructions are the relevant comparison for repeated experiments/process restarts.
- This notebook focuses on moving **one-time preprocessing** off TensorFlow graph ops to reduce allocator pressure during setup.
